# Kmeans

In [2]:
import requests
import base64
import pandas as pd
from typing import Dict, Any
from dotenv import load_dotenv
load_dotenv()
import os
import json
from typing import Dict, Any

In [3]:
def folder_to_df_merged(folder_path: str) -> Dict[str, pd.DataFrame]:
    result = {}

    for file in os.listdir(folder_path):
        if not file.endswith(".json"):
            continue

        path = os.path.join(folder_path, file)

        with open(path, encoding="utf-8") as f:
            data: Any = json.load(f)

        if isinstance(data, list) and all(isinstance(i, dict) for i in data):
            df = pd.json_normalize(data)

        elif isinstance(data, dict):
            list_keys = [k for k, v in data.items() if isinstance(v, list)]

            if list_keys:
                main_key = list_keys[0]
                df = pd.json_normalize(
                    data,
                    record_path=main_key,
                    meta=[k for k in data.keys() if k != main_key],
                    errors="ignore"
                )
            else:
                df = pd.DataFrame([data])

        else:
            df = pd.DataFrame()

        result[file.replace(".json", "")] = df
    
    return pd.concat(result.values(), ignore_index=True)

df = folder_to_df_merged("../data/raw")

df_result = df.explode("genre").dropna(subset=['genre'])


In [4]:
df_result = df_result.replace({'True': 1, 'False': 0})
df_result

,ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,...,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode,user,artist_id,genre
2,2023-11-03T19:08:49Z,windows,114865,BR,200.185.254.31,Costa Rica (feat. The Kid LAROI) - Remix,Bankrol Hayden,Pain is Temporary,spotify:track:3tdjTdCCgKtwacsICCtPZZ,None,...,clickrow,trackdone,False,False,False,1.699038e+09,False,gigi,0Yr4BBpK2dkCp2UsrJ9LZN,melodic rap
4,2023-11-03T19:12:12Z,windows,90010,BR,191.23.42.158,Blueberry Faygo,Lil Mosey,Blueberry Faygo,spotify:track:6wJYhPfqk3KGhHRG76WzOh,None,...,playbtn,unexpected-exit,False,False,False,1.699039e+09,False,gigi,6Xgp2XMz1fhVYe7i6yNAax,melodic rap
5,2023-11-03T21:44:28Z,windows,159129,BR,191.23.42.158,Blueberry Faygo,Lil Mosey,Blueberry Faygo,spotify:track:6wJYhPfqk3KGhHRG76WzOh,None,...,appload,trackdone,False,False,False,1.699047e+09,False,gigi,6Xgp2XMz1fhVYe7i6yNAax,melodic rap
6,2023-11-03T21:47:12Z,windows,162546,BR,191.23.42.158,Blueberry Faygo,Lil Mosey,Blueberry Faygo,spotify:track:6wJYhPfqk3KGhHRG76WzOh,None,...,trackdone,trackdone,False,False,False,1.699048e+09,False,gigi,6Xgp2XMz1fhVYe7i6yNAax,melodic rap
7,2023-11-03T21:49:55Z,windows,162546,BR,191.23.42.158,Blueberry Faygo,Lil Mosey,Blueberry Faygo,spotify:track:6wJYhPfqk3KGhHRG76WzOh,None,...,trackdone,trackdone,False,False,False,1.699048e+09,False,gigi,6Xgp2XMz1fhVYe7i6yNAax,melodic rap
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319635,2026-01-20T03:44:09Z,ios,0,BR,2804:14d:328c:56a8:804a:e6b5:cc3:18b5,You Make Me Feel So Young - Live At Royal Fest...,Frank Sinatra,You Make Me Feel So Young,spotify:track:6OQPeBd6CKouqH0Dsf1XkG,None,...,unknown,endplay,False,True,False,1.768881e+09,False,Fefo,1Mxqyy3pSjf8kZZL4QVxS0,swing music
319635,2026-01-20T03:44:09Z,ios,0,BR,2804:14d:328c:56a8:804a:e6b5:cc3:18b5,You Make Me Feel So Young - Live At Royal Fest...,Frank Sinatra,You Make Me Feel So Young,spotify:track:6OQPeBd6CKouqH0Dsf1XkG,None,...,unknown,endplay,False,True,False,1.768881e+09,False,Fefo,1Mxqyy3pSjf8kZZL4QVxS0,jazz
319635,2026-01-20T03:44:09Z,ios,0,BR,2804:14d:328c:56a8:804a:e6b5:cc3:18b5,You Make Me Feel So Young - Live At Royal Fest...,Frank Sinatra,You Make Me Feel So Young,spotify:track:6OQPeBd6CKouqH0Dsf1XkG,None,...,unknown,endplay,False,True,False,1.768881e+09,False,Fefo,1Mxqyy3pSjf8kZZL4QVxS0,vocal jazz
319639,2026-01-25T05:28:54Z,ios,0,BR,2804:38a:a09f:e5c5:3c84:35f8:2615:f6ba,Vou Deitar E Rolar (Quaquaraquaqua),Elis Regina,Vou Deitar E Rolar (Quaquaraquaqua),spotify:track:0GjOplhCphC5IF8ZHaXjbq,None,...,unknown,endplay,False,True,False,1.769319e+09,False,Fefo,7dnT2FUXhjirperXaH22IJ,mpb


In [5]:
df_result.columns

Index(['ts', 'platform', 'ms_played', 'conn_country', 'ip_addr',
       'master_metadata_track_name', 'master_metadata_album_artist_name',
       'master_metadata_album_album_name', 'spotify_track_uri', 'episode_name',
       'episode_show_name', 'spotify_episode_uri', 'audiobook_title',
       'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title',
       'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline',
       'offline_timestamp', 'incognito_mode', 'user', 'artist_id', 'genre'],
      dtype='object')

In [6]:
df_result = df_result.drop(columns = ['platform','conn_country', 'ip_addr','spotify_track_uri', 'episode_name',
       'episode_show_name', 'spotify_episode_uri', 'audiobook_title',
       'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title',"master_metadata_track_name", "master_metadata_album_artist_name", "master_metadata_album_album_name", "offline", "incognito_mode","offline_timestamp", "artist_id"])

In [7]:
df_result['ms_played'] = pd.to_numeric(df_result['ms_played'], errors='coerce')

In [8]:
pivot = (
    df_result
    .groupby(['genre', 'user'])
    .agg(
        sum_ms_played = pd.NamedAgg("ms_played","sum"),
        mean_ms_played = pd.NamedAgg("ms_played","mean"),
        skipped_percent = pd.NamedAgg("skipped","mean")
    )
)

In [9]:
pivot = pivot.unstack("user").fillna(0)
pivot.columns = [
    f"{pessoa}_{valor}" 
    for valor, pessoa in pivot.columns
]

In [10]:
from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()
x_scaled = scaler.fit_transform(pivot)

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

def encontrar_cotovelo_kmeans(
    df,
    k_min=1,
    k_max=30,
    random_state=42
):

    X = df.copy()


    inertias = []
    k_range = range(k_min, k_max + 1)

    for k in k_range:
        model = KMeans(n_clusters=k, random_state=random_state, n_init="auto")
        model.fit(X)
        inertias.append(model.inertia_)

    x = np.array(list(k_range))
    y = np.array(inertias)

    p1 = np.array([x[0], y[0]])
    p2 = np.array([x[-1], y[-1]])

    distances = []
    for i in range(len(x)):
        p = np.array([x[i], y[i]])
        distance = np.abs(np.cross(p2 - p1, p1 - p)) / np.linalg.norm(p2 - p1)
        distances.append(distance)

    elbow_index = np.argmax(distances)
    k_otimo = x[elbow_index]

    return k_otimo

In [12]:
k_otimo = encontrar_cotovelo_kmeans(x_scaled)
print(k_otimo)
kmeans = KMeans(n_clusters=k_otimo, random_state=42)
pivot['cluster'] = kmeans.fit_predict(x_scaled)

11


In [13]:
pivot

,Allanabre_sum_ms_played,Cury_sum_ms_played,Fefo_sum_ms_played,Gueguelas_sum_ms_played,gigi_sum_ms_played,Allanabre_mean_ms_played,Cury_mean_ms_played,Fefo_mean_ms_played,Gueguelas_mean_ms_played,gigi_mean_ms_played,Allanabre_skipped_percent,Cury_skipped_percent,Fefo_skipped_percent,Gueguelas_skipped_percent,gigi_skipped_percent,cluster
genre,,,,,,,,,,,,,,,,
acid jazz,0.0,2377166.0,0.0,0.0,0.0,0.000000,216106.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9
acid rock,0.0,0.0,48562.0,0.0,0.0,0.000000,0.000000,24281.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9
acid techno,0.0,0.0,0.0,215282.0,401230.0,0.000000,0.000000,0.000000,53820.500000,57318.571429,0.000000,0.000000,0.000000,0.750000,0.428571,0
acoustic pop,301836.0,0.0,79409.0,186320.0,0.0,150918.000000,0.000000,79409.000000,93160.000000,0.000000,0.500000,0.000000,1.000000,0.000000,0.000000,1
adult standards,4223668.0,10982.0,54382489.0,522269.0,11672.0,117324.111111,5491.000000,126470.904651,19343.296296,2918.000000,0.222222,0.500000,0.462791,0.111111,0.750000,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
west coast hip hop,437131.0,2043514.0,14323455.0,563289258.0,49734075.0,145710.333333,145965.285714,130213.227273,141458.879458,158894.808307,0.333333,0.285714,0.427273,0.256153,0.255591,6
witch house,2543988.0,197940.0,9141238.0,2459748.0,103346.0,158999.250000,197940.000000,103877.704545,64730.210526,12918.250000,0.125000,1.000000,0.090909,0.236842,0.875000,6
worship,465320.0,0.0,74680.0,1630.0,0.0,232660.000000,0.000000,18670.000000,543.333333,0.000000,0.000000,0.000000,0.250000,0.000000,0.000000,9


In [14]:
genero = input("Escolha um genero")

try:
    cluster = pivot.loc[[genero]]["cluster"][0]
    print("sugestoes:")
    generos_sugerids = pivot[pivot["cluster"] == cluster].index.tolist()

    for x in generos_sugerids:
        if x != genero:
            print(x) 
except:
    print("genero nao existe")





genero nao existe


# KNN

In [15]:
# Cria um score, "penalizando" skip/forward
df_result["score"] = (
    df_result["ms_played"]
    * (1 - df_result["skipped"])
    * (df_result["reason_end"] != "forward")
)
df_result = df_result[df_result["score"] != 0]
df_result

,ts,ms_played,reason_start,reason_end,shuffle,skipped,user,genre,score
2,2023-11-03T19:08:49Z,114865,clickrow,trackdone,False,False,gigi,melodic rap,114865
4,2023-11-03T19:12:12Z,90010,playbtn,unexpected-exit,False,False,gigi,melodic rap,90010
5,2023-11-03T21:44:28Z,159129,appload,trackdone,False,False,gigi,melodic rap,159129
6,2023-11-03T21:47:12Z,162546,trackdone,trackdone,False,False,gigi,melodic rap,162546
7,2023-11-03T21:49:55Z,162546,trackdone,trackdone,False,False,gigi,melodic rap,162546
...,...,...,...,...,...,...,...,...,...
319630,2025-12-30T02:45:34Z,141302,trackdone,trackdone,False,False,Fefo,uk grime,141302
319631,2025-12-30T03:04:26Z,261303,trackdone,trackdone,False,False,Fefo,uk drill,261303
319631,2025-12-30T03:04:26Z,261303,trackdone,trackdone,False,False,Fefo,grime,261303
319631,2025-12-30T03:04:26Z,261303,trackdone,trackdone,False,False,Fefo,uk grime,261303


In [16]:
# Cria um score, "penalizando" skip/forward
df_result["score"] = (
    df_result["ms_played"]
    * (1 - df_result["skipped"])
    * (df_result["reason_end"] != "forward")
)
df_result = df_result[df_result["score"] != 0]
df_result

,ts,ms_played,reason_start,reason_end,shuffle,skipped,user,genre,score
2,2023-11-03T19:08:49Z,114865,clickrow,trackdone,False,False,gigi,melodic rap,114865
4,2023-11-03T19:12:12Z,90010,playbtn,unexpected-exit,False,False,gigi,melodic rap,90010
5,2023-11-03T21:44:28Z,159129,appload,trackdone,False,False,gigi,melodic rap,159129
6,2023-11-03T21:47:12Z,162546,trackdone,trackdone,False,False,gigi,melodic rap,162546
7,2023-11-03T21:49:55Z,162546,trackdone,trackdone,False,False,gigi,melodic rap,162546
...,...,...,...,...,...,...,...,...,...
319630,2025-12-30T02:45:34Z,141302,trackdone,trackdone,False,False,Fefo,uk grime,141302
319631,2025-12-30T03:04:26Z,261303,trackdone,trackdone,False,False,Fefo,uk drill,261303
319631,2025-12-30T03:04:26Z,261303,trackdone,trackdone,False,False,Fefo,grime,261303
319631,2025-12-30T03:04:26Z,261303,trackdone,trackdone,False,False,Fefo,uk grime,261303


In [19]:
pivot = (
    df_result.groupby(["user", "genre"])
      .agg(
          score=("score", "mean")
      )
      .reset_index()
).pivot(
    index="user",
    columns="genre",
    values="score"
).fillna(0)

pivot

genre,acid jazz,acid rock,acid techno,acoustic pop,adult standards,afro house,afro r&b,afro soul,afro tech,afro-cuban jazz,...,vietnamese bolero,vietnamese lo-fi,visual kei,vocal jazz,vocaloid,west coast hip hop,witch house,worship,yacht rock,zouk
user,,,,,,,,,,,,,,,,,,,,,
Allanabre,0.0,0.0,0.00,292566.0,138290.571429,0.000000,28001.000000,0.000000,0.0,0.0,...,0.0,0.000000,2501.000000,138290.571429,239410.888889,146921.500000,193440.230769,232660.000000,109077.586207,0.0
Cury,216106.0,0.0,0.00,0.0,10868.000000,0.000000,0.000000,0.000000,0.0,0.0,...,4475.0,0.000000,0.000000,0.000000,0.000000,188481.800000,0.000000,0.000000,256022.000000,0.0
Fefo,0.0,24281.0,0.00,0.0,159925.393939,0.000000,124003.333333,95794.606061,0.0,157081.0,...,0.0,129554.666667,0.000000,177213.442857,210102.000000,180781.809524,112803.812500,23533.333333,150465.000000,0.0
Gueguelas,0.0,0.0,212758.00,93160.0,33191.400000,3054.500000,105402.142857,0.000000,0.0,0.0,...,0.0,0.000000,65390.272727,37976.294118,5446.666667,181255.603509,76321.642857,543.333333,38000.000000,0.0
gigi,0.0,0.0,93064.25,0.0,1258.000000,101582.428571,130591.000000,0.000000,61648.0,0.0,...,0.0,0.000000,0.000000,1258.000000,0.000000,196555.480687,92536.000000,0.000000,177595.428571,253228.0


In [20]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

knn.fit(pivot)

NearestNeighbors(algorithm='brute', metric='cosine')

In [21]:
distancias, indices = knn.kneighbors(
    pivot.loc[["gigi"]],
    n_neighbors=3  # com 6 usuários, 2 ou 3 é ideal
)

usuarios_parecidos = pivot.index[indices.flatten()][1:] 
similaridades = 1 - distancias.flatten()[1:] 

for u, s in zip(usuarios_parecidos, similaridades):
    print(u, round(s, 3))


Cury 0.529
Gueguelas 0.5


In [22]:
import numpy as np

sim_vizinhos = similaridades
matriz_vizinhos = pivot.loc[usuarios_parecidos]

media_ponderada = np.average(
    matriz_vizinhos,
    axis=0,
    weights=sim_vizinhos
)

media_ponderada = pd.Series(
    media_ponderada,
    index=pivot.columns
)

usuario_vector = pivot.loc["Fefo"]

recomendacao = (media_ponderada - usuario_vector)

recomendacao = recomendacao.sort_values(ascending=False)

In [23]:
top5 = recomendacao.head(5)
print(top5)

genre
doom metal      325574.253630
stoner rock     324366.363526
thrash metal    317937.320267
art rock        241455.477260
heavy metal     232726.504005
dtype: float64
